# Interactive Dashboard Report

## Introduction

With this assignment allowing for the use of generative AI in assiting in creating of the product, I was interested in exploring how energy generation has changed in recent years.  From there take a look at how the breakdown of the energy generation has evolved (solar, coal, gas, etc).  Then see how each country compares to each other, possibly explore emissions and how the share of the production has shifted in regards to renweables.

**Finished dashboard**

<img src="./image_1794842890782.png" width="50%">

*Live version:* https://siads-521-energy-dashboard-7474654444631410.aws.databricksapps.com/ (there may be issues with permissions with this)

*Alternate:* https://siads-521-energy-dashboard-c3d248ea.plotly.app/

*Video Walkthrough:* https://drive.google.com/file/d/11idJElFo-3g9K5-RxpFDbCMR2mbwHmXL/view?usp=sharing

*git repo:* https://github.com/ncpeot/portfolio/tree/main/SIADS_521_Assignment3

In [0]:
#%pip install dash
import pandas as pd
from dash import dcc, html
import plotly.express as px
import plotly.graph_objects as go

## Library choice and fit (15 pts)

**The library: Plotly + Dash**

- **Created by:** Plotly (the company behind Plotly.js and Plotly Express)
- **License:** MIT - allows commercial use, modification, distribution, and private use without restriction.
- **How to install:** `%pip install dash`
- **Declarative or procedural:** Dash is primarily declarative but does rely on standard python for processing data.  You define a tree of Python objects (html.Div, dcc.Graph, etc) and declare relationshipts using callbacks.

**Jupyter integration and limitations**

- Jupyter Integration: Dash runs by starting its dev server hosted locally (http://127.0.0.1:8050) and allows you to open via browswer.
- Limitations hit in this project: Notebook was developed on databricks so once getting to the testing of the dash app, was unable to run and test via local version.  Either had to commit changes to git, sync to VS Code locally and test or fully deploy and test.  Both took a decent amount of time.

**Why Dash fits this dataset**

The dataset has one row per country-year with a fixed set of metrics.  This allows for an easy use of interactive filters that ineract easily with the visualizations.

**Alternative considered and rejected:** Streamlit. Its model reruns the whole script top-to-bottom on every widget change, so each slider tweak would redo all data prep and rebuild all four figures.  Dash's callbacks let me update only what actually depends on the changed input. Dash also reuses my Plotly Express figures directly, with no conversion layer.

## Data

- **Source:** the Our World in Data **Energy** dataset (`owid_energy.csv`), which OWID compiles from the Energy Institute *Statistical Review of World Energy* and related sources - https://ourworldindata.org/energy
- **License:** CC BY 4.0 (Creative Commons Attribution) - the dataset can be shared, adapted, and built upon, including commercially, as long as Our World in Data is credited.
- **Size (raw):** 19,494 rows × 130 columns (9.1 MB)

**Columns used**

- `country` - name of the country or aggregate region (string)
- `iso_code` - ISO country code; missing for aggregates such as "World" (string)
- `year` - calendar year of the observation (integer)
- `bioenergy_electricity_twh`, `coal_electricity_twh`, `gas_electricity_twh`, `hydro_electricity_twh`, `nuclear_electricity_twh`, `oil_electricity_twh`, `solar_electricity_twh`, `wind_electricity_twh`, `other_renewables_excluding_bioenergy_electricity_twh` - electricity generation by source, in terawatt-hours (float)
- `renewables_electricity_share_pct` - share of electricity generated from renewables, % (float)
- `fossil_fuels_electricity_share_pct` - share of electricity generated from fossil fuels, % (float)
- `electricity_generation_twh` - total electricity generation in terawatt-hours; used to rank the dashboard's default selection of countries (float)
- `electricity_generation_per_capita_kwh` - electricity generated per person, kWh(float)
- `electricity_emissions_mtco2eq` - greenhouse-gas emissions from electricity generation, MtCO2eq (float)

### Cleaning steps
- Step 1 - Remove all entries with no ISO code.  These were regional aggregates such as World, North America, Europe, etc.
- Step 2 - Filter for data between 1985 and 2024 (inclusive).  Most data is missing prior to 1985 and 2025 contains sparse data for a lot of countries
- Step 3 - Remove all unused columns

In [0]:
#data import and cleaning
df = pd.read_csv('owid_energy.csv')

#define columns to be used
columns = [
    'country',
    'iso_code',
    'year',
    'bioenergy_electricity_twh',
    'coal_electricity_twh',
    'gas_electricity_twh',
    'hydro_electricity_twh',
    'nuclear_electricity_twh',
    'oil_electricity_twh',
    'solar_electricity_twh',
    'wind_electricity_twh',
    'other_renewables_excluding_bioenergy_electricity_twh',
    'renewables_electricity_share_pct',
    'fossil_fuels_electricity_share_pct',
    'electricity_generation_twh',
    'electricity_generation_per_capita_kwh',
    'electricity_emissions_mtco2eq',
]

#get list of all countries by name with no iso code
countries = df[df['iso_code'].notna()]['country'].unique().tolist()

#data filtering
#Data prior to 1985 is missing values and 2025 is sparse, 
#remove any countries that have no ISO code
mask = (
    df['year'].isin(range(1985,2025)) &
    df['country'].isin(countries)
    )

df = df[mask][columns]

## Visualization techniques (20 pts)

- **Line Chart (fig1)** - Used here to see the trends in energy generation per capita for each country in a easy to compare way. Avoid when there are many series, the x-axis is not an ordered continuous quantity, or if comparing data at a single point in time.

- **Stacked Area Chart (fig2)** - shows the total generation of the selected countries and its composition by source over time.  Used to see how the mix of the selected countries may be changing (increasing solar).  This chart should be avioded when many categories are tiny, if the values may be negative or if there's no relational value in showing cattegories stacked like this.

- **Choropleth (fig3)** - shows a single metric by geography; color encodes the share of generation from renewables, 0-100% on a fixed scale. Allows visual comparison of regions that are prioritizing renewable energy for the last year of the selection.  Avoid when precise comparisons are needed or when the metric is a raw total.

- **Slope Chart (fig5)** - A slope chart is a line chart reduced to exactly two time points and used as vertical axes with one line per country connecting its renewables share at each endpoint.  It distills the whole selected range of years into a figure representing the net change in renewable energy generation share, so the slope of the line gives a easy to compare visual - what country increased renewable production heaviliy, or ones that declined.  While similar to a standard line chart, this allows a quick, at-a-glance comparsion of the change.

- **Parallel Coordinates (fig4)** - I kept this chart in the code as it was the original option for the new-to-class method but the plot was not able to easily show country comparisons as I could not find a method to color/label the lines with ease to distinguish each country.  This would ideally have been used to show trade-offs, high generation from renewables leading to lower emissions.

**How the charts complement each other**

The layout follows the natural flow of the questions. Row 1 (line + area) showing the countries energy generation per capita and the breakdown of where it comes from (total of all countries selected).  Row 2 pairs the map with the slope chart, show where each contry stands relative to one another in their renewable generation with the slope chart giving a quick overview of how the selected countries renewables have changed over the selected period. 

## Building and exploring (35 pts)

Build the dashboard step by step: basic components first, then layout, then linked interactivity. Include a code snippet **and a screenshot** at each key stage.

### Step 1 - Basic components

In [0]:
#Figure 1 - Line Chart
def generate_country_data(df: pd.DataFrame(), countries: list = ['World'], years: list = range(1985,2025)) -> pd.DataFrame():
    dfc = df.copy()
    result = dfc[dfc['country'].isin(countries) & dfc['year'].isin(years)].rename(columns = {'electricity_generation_per_capita_kwh': 'energy generation'})

    return result

data = generate_country_data(df, ['United States', 'United Kingdom'])

# figure 1
fig1 = px.line(
    data,
    x = 'year',
    y = 'energy generation',
    color = 'country',
    markers = True,
    title = f'Total Energy Generation per Capita by Country ({min(data['year'])}-{max(data['year'])})',
    hover_data = {
        'year': False, 
        'country': True, 
        'energy generation': ":,.0f"},
    )

fig1.update_layout(
    title_x = 0.5,
    height = 650,
    width = 1000,
    plot_bgcolor = 'rgba(0,0,0,0)',
    paper_bgcolor = 'rgba(0,0,0,0)',
    xaxis_title = 'Year',
    yaxis_title = 'Total Energy (kWh)',
    legend_title ='Country',
    hovermode = 'x unified',
)

#black axis lines so the axes stay visible on the transparent background
fig1.update_xaxes(showline = True, linecolor = 'black', tickcolor = 'black')
fig1.update_yaxes(showline = True, linecolor = 'black', tickcolor = 'black')

fig1.show()

In [0]:
#Figure 2 - Stacked Area Chart

def generate_energy_share(df: pd.DataFrame(), countries: list = ['World'], years: list = range(1985,2025)) -> tuple[pd.DataFrame(), list]:
    dfc = df.copy()

    #define and get columns we want
    columns = [
        'country',
        'year',
        'bioenergy_electricity_twh',
        'coal_electricity_twh',
        'gas_electricity_twh',
        'hydro_electricity_twh',
        'nuclear_electricity_twh',
        'oil_electricity_twh',
        'solar_electricity_twh',
        'wind_electricity_twh',
        'other_renewables_excluding_bioenergy_electricity_twh',
        ]

    #create dataframe and remove suffixes and fix column names for display
    result = dfc[columns].rename(columns = lambda x: x.removesuffix('_electricity_twh')).rename(columns = {'other_renewables_excluding_bioenergy': 'other renewables'})

    #convert to long format
    result = pd.melt(result, id_vars = ['country', 'year'], var_name = 'energy_type', value_name = 'values')

    #filter to the selected countries and sum their values for each year and energy type
    result = (result[result['country'].isin(countries) & result['year'].isin(years)]
              .groupby(['year', 'energy_type'], as_index = False)['values'].sum())

    #create sort order based on values for final plot year
    order = result[result['year'] == dfc['year'].max()].sort_values('values', ascending = False)['energy_type'].to_list()

    return result, order

data, order = generate_energy_share(df, countries = ['United States', 'United Kingdom'])

#figure2
fig2 = px.area(
    data,
    x = 'year',
    y = 'values',
    color = 'energy_type',
    line_group = 'energy_type',
    category_orders = {'energy_type': order}, #order of plotting is based on values of right most year being plotted with largest at the bottom
    title = f'Total Energy Generated by Source ({min(data['year'])}-{max(data['year'])})',
    hover_data = {
        'year': False,
        'energy_type': True,
        'values': ":,.0f"
    }
)

fig2.update_layout(
    title_x = 0.5,
    height = 650,
    width = 1000,
    plot_bgcolor = 'rgba(0,0,0,0)',
    paper_bgcolor = 'rgba(0,0,0,0)',
    xaxis_title = 'Year',
    yaxis_title = 'Total Energy Generated (TWh)',
    legend_title ='Energy Source',
    hovermode = 'x unified'
)

#black axis lines so the axes stay visible on the transparent background
fig2.update_xaxes(showline = True, linecolor = 'black', tickcolor = 'black')
fig2.update_yaxes(showline = True, linecolor = 'black', tickcolor = 'black')

fig2.show()

In [0]:
#Figure 3 - Choropleth Map

def generate_renewable_energy(df: pd.DataFrame(), years: list = range(1985,2025)) -> pd.DataFrame():
    #copy dataframe and restrict to most recent year
    dfc = df[df['year'] == max(years)].copy()

    #drop columns and na values from iso_code and renewables_electricity_share_pct
    result = dfc[['country','iso_code','year', 'renewables_electricity_share_pct']].dropna(subset = ['iso_code', 'renewables_electricity_share_pct']).rename(columns = {'renewables_electricity_share_pct': 'renewables_share'})

    return result

data = generate_renewable_energy(df)

#figure 3
fig3 = px.choropleth(
    data,
    locations = 'iso_code',
    locationmode = 'ISO-3',
    color = 'renewables_share',
    hover_name = 'country',
    hover_data = {
        'iso_code': False,
        'year': True,
        'renewables_share': ":.0f"
    },
    color_continuous_scale = 'RdYlGn',
    range_color = [0, 100],
    title = f'Renewable Energy Generation by Country ({max(data['year'])})'
)

fig3.update_layout(
    title_x = 0.5,
    height = 650,
    width = 1000,
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    coloraxis_colorbar=dict(
        title="Renewable<br>Generation",
        ticksuffix="%"
    )
)

fig3.show()

In [0]:
#Figure 4 - Paralell Coordinates Plot (replaced with Figure 5)

def generate_profiles(df: pd.DataFrame(), countries: list = ['World'], years: list = range(1985,2025)) -> pd.DataFrame():
    dfc = df.copy()

    #define columns required
    columns = [
        'country',
        'year',
        'electricity_generation_per_capita_kwh',
        'renewables_electricity_share_pct',
        'fossil_fuels_electricity_share_pct',
        'electricity_emissions_mtco2eq',
    ]

    #select required columns and rename
    result = (dfc[columns].rename(columns = {
        'electricity_generation_per_capita_kwh': 'energy_per_capita', 
        'renewables_electricity_share_pct': 'renewables_share', 
        'fossil_fuels_electricity_share_pct': 'fossil_fuels_share', 
        'electricity_emissions_mtco2eq': 'emissions'}))
    
    #filter to countries and years, drop any that have na
    result = (result[(result['country'].isin(countries)) & (result['year'] == max(years))]
              .dropna(subset = ['energy_per_capita','renewables_share', 'fossil_fuels_share', 'emissions']))

    return result

#test values
data = generate_profiles(df, countries = ['United States', 'China','United Kingdom'])

#figure 4
fig4 = px.parallel_coordinates(
    data,
    dimensions = [
        'energy_per_capita',
        'renewables_share',
        'fossil_fuels_share',
        'emissions'
    ],
    color = 'renewables_share',
    color_continuous_scale = 'RdYlGn',
    range_color = [0, 100],
    labels = {
        'energy_per_capita': 'Energy per capita (kWh)',
        'renewables_share': 'Renewables share (%)',
        'fossil_fuels_share': 'Fossil fuels share (%)',
        'emissions': 'Emissions (MtCO2eq)'
    },
    title = f'Energy Profiles of Countries ({max(data['year'])})'
)

fig4.update_layout(
    title_x=0.5,
    height=650,
    width = 1000,    
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    coloraxis_colorbar=dict(
        title="Renewable<br>Share",
        ticksuffix="%"
    )
)

fig4.show()

In [0]:
#Figure 5 - Slope Chart

def generated_differential(df: pd.DataFrame(), countries: list = ['World'], years: list = range(1985,2025)) -> pd.DataFrame():
    dfc = df.copy()

    #define columns required
    columns = [
        'country',
        'year',
        'renewables_electricity_share_pct'
        ]
    
    #filter for countries and years
    result = (dfc[columns][(dfc['country'].isin(countries)) & (dfc['year'].isin([min(years), max(years)]))])

    #pivot to get min and max on separate columns, assume nulls would be 0 (this may be a problem if end year is null)
    result = result.pivot(index = 'country', columns = 'year', values = 'renewables_electricity_share_pct').reset_index().fillna(0)

    return result

data = generated_differential(df, countries = ['United States', 'China', 'Canada', 'Brazil', 'France' ])

#prompt: is there a way to prevent the text labels on fig5 from overlapping when their end points are close?
#result:
#dodge overlapping labels - plotly has no built-in label repulsion, so cluster
#close endpoints and spread their labels to keep a minimum vertical gap
gap = 4 #minimum label separation in %-points (~19px at this chart height)
end_col = data.columns[2]
pts = data[['country', end_col]].sort_values(end_col, ascending = False).reset_index(drop = True)
label_y = {}
i = 0
while i < len(pts):
    #grow the cluster while its span is too tight to fit all labels with gaps
    j = i
    while j + 1 < len(pts) and pts.loc[i, end_col] - pts.loc[j + 1, end_col] < gap * (j + 1 - i):
        j += 1
    cluster = pts.loc[i:j]
    #spread the cluster's labels evenly, centered on the cluster's midpoint
    mid = (pts.loc[i, end_col] + pts.loc[j, end_col]) / 2
    top = min(100, mid + gap * (len(cluster) - 1) / 2)
    top = max(top, gap * (len(cluster) - 1)) #keep the spread inside the 0-100 axis range
    for k, (_, r) in enumerate(cluster.iterrows()):
        label_y[r['country']] = top - k * gap
    i = j + 1

#figure 5
fig5 = go.Figure()

#plot each line
for _, row in data.iterrows():
    fig5.add_trace(
        go.Scatter(
        x = [0, 1],
        y = [row[data.columns[1]], row[data.columns[2]]],
        mode = 'lines+markers',
        name = row['country'],
        cliponaxis = False,
        )
    )

#prompt: let's also nudge the labels slightly right, away from the axis as they look way too close
#result:
#country labels at the (possibly nudged) right endpoints - annotations render
#in the margin layer, take a pixel offset, and hover ignores them
for _, row in data.iterrows():
    fig5.add_annotation(
        x = 1,
        y = label_y[row['country']],
        xref = 'x',
        yref = 'y',
        text = row['country'],
        showarrow = False,
        xanchor = 'left',
        xshift = 8, #nudge labels slightly right, away from the axis line
        font = dict(color = 'black')
    )

#ad the vertical line for the right axis
fig5.add_shape(
    type = 'line',
    xref = 'x', 
    yref = 'paper',
    x0 = 1, 
    y0 = 0,
    x1 = 1, 
    y1 = 1,
    line = dict(color = 'black', width = 1)
)

#title must precede title_x - a title kwarg replaces the title object and would drop its x
fig5.update_layout(
    title=(
        f"Change in Renewable Energy Share "
        f"({data.columns[1]}-{data.columns[2]})"
    ),
    title_x=0.5,
    height=650,
    width = 1000,
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    showlegend=False,
    margin=dict(
        l=80,
        r=160,
        t=100,
        b=70
    ),
    xaxis = dict(
        tickmode = 'array',
        tickvals = [0, 1],
        ticktext = [data.columns[1], data.columns[2]],
        showgrid = False,
        showline = False,
        ticks = 'outside',
        ticklen = 15,
        tickcolor = 'rgba(0,0,0,0)', #invisible tick marks push the year labels away from the axis
        range = [0, 1]
    ),
    yaxis = dict(
        title = 'Renewable Energy Share (%)',
        range = [0, 100],
        ticksuffix = '%',
        dtick = 20,
        showline = True,
        linecolor = 'black',
        tickcolor = 'black',
    )
)

fig5.show()

### Step 2 - Layout

In [0]:
#prompt:    Using the code in this notebook, generate a dash app using the following layout:
#           Row 1: Fig1, Fig2
#           Row 2: Fig3, Fig5

#result:
# Step 2 - Layout: assemble the four figures into the Dash app
from dash import Dash, dcc, html
from threading import Thread

app = Dash(__name__)

# Row 1 - fig1 (energy per capita) and fig2 (generation by source) side by side
row1 = html.Div(
    [
        dcc.Graph(figure=fig1, style={'flex': 1}),
        dcc.Graph(figure=fig2, style={'flex': 1}),
    ],
    style={'display': 'flex', 'gap': '10px'},
)

# Row 2 - fig3 (renewables choropleth) and fig5 (change in renewables share) side by side
row2 = html.Div(
    [
        dcc.Graph(figure=fig3, style={'flex': 1}),
        dcc.Graph(figure=fig5, style={'flex': 1}),
    ],
    style={'display': 'flex', 'gap': '10px'},
)

app.layout = html.Div([row1, row2])

# Serve the app on a background thread so the notebook stays responsive
Thread(target=lambda: app.run(host='0.0.0.0', port=8050), daemon=True).start()
print('Dashboard serving at http://127.0.0.1:8050')

<img src="./image_1794854783869.png" width="50%">

### Step 3 - Linked interactivity


In [0]:
#prompt: Add callbacks to link the figures
#result:

# # Step 3 - Linked interactivity: shared country/year controls drive all four figures
from dash import Dash, dcc, html
from dash.dependencies import Input, Output, State
from dash.exceptions import PreventUpdate
from threading import Thread
import plotly.graph_objects as go

def empty_figure(title, message):
    fig = go.Figure()
    fig.update_layout(
        title = title,
        title_x = 0.5,
        annotations = [dict(text = message, showarrow = False, x = 0.5, y = 0.5,
                            xref = 'paper', yref = 'paper', font = dict(size = 16))]
    )
    return fig

# Figure builders - the Step 1 chart code, parameterized by the shared controls
def make_fig1(countries, years):
    data = generate_country_data(df, countries = countries, years = years)
    if data.empty:
        return empty_figure('Total Energy Generation per Capita', 'No data for the selected countries')
    fig = px.line(
        data,
        x = 'year',
        y = 'energy generation',
        color = 'country',
        markers = True,
        title = f"Total Energy Generation per Capita by Country ({min(data['year'])}-{max(data['year'])})",
        hover_data = {
            'year': False,
            'country': True,
            'energy generation': ':,.0f'
        },
    )
    fig.update_layout(
        title_x = 0.5,
        height = 650,
        plot_bgcolor = 'rgba(0,0,0,0)',
        paper_bgcolor = 'rgba(0,0,0,0)',
        xaxis_title = 'Year',
        yaxis_title = 'Total Energy Generated (kWh)',
        legend_title = 'Country',
        hovermode = 'x unified'
    )
    #black axis lines so the axes stay visible on the transparent background
    fig.update_xaxes(showline = True, linecolor = 'black', tickcolor = 'black')
    fig.update_yaxes(showline = True, linecolor = 'black', tickcolor = 'black')
    return fig

def make_fig2(countries, years):
    data, order = generate_energy_share(df, countries = countries, years = years)
    if data.empty:
        return empty_figure('Energy Generation by Source', 'No data for the selected countries')
    # order the stack by the right-most year actually being plotted
    order = data[data['year'] == max(years)].sort_values('values', ascending = False)['energy_type'].to_list()
    fig = px.area(
        data,
        x = 'year',
        y = 'values',
        color = 'energy_type',
        line_group = 'energy_type',
        category_orders = {'energy_type': order},
        title = f"Total Energy Generated by Source ({min(data['year'])}-{max(data['year'])})",
        hover_data = {
            'year': False,
            'energy_type': True,
            'values': ':,.0f'
        },
    )
    fig.update_layout(
        title_x = 0.5,
        height = 650,
        plot_bgcolor = 'rgba(0,0,0,0)',
        paper_bgcolor = 'rgba(0,0,0,0)',
        xaxis_title = 'Year',
        yaxis_title = 'Total Energy Generated (TWh)',
        legend_title = 'Energy Source',
        hovermode = 'x unified'
    )
    #black axis lines so the axes stay visible on the transparent background
    fig.update_xaxes(showline = True, linecolor = 'black', tickcolor = 'black')
    fig.update_yaxes(showline = True, linecolor = 'black', tickcolor = 'black')
    return fig

def make_fig3(year):
    data = generate_renewable_energy(df, years = [year])
    if data.empty:
        return empty_figure(f'Renewable Energy Generation ({year})', 'No data for this year')
    fig = px.choropleth(
        data,
        locations = 'iso_code',
        locationmode = 'ISO-3',
        color = 'renewables_share',
        hover_name = 'country',
        hover_data = {'iso_code': False, 'year': True, 'renewables_share': ':.0f'},
        color_continuous_scale = 'RdYlGn',
        range_color = [0, 100],
        title = f'Renewable Energy Generation by Country ({year})'
    )
    fig.update_layout(
        title_x = 0.5,
        height = 650,
        plot_bgcolor = 'rgba(0,0,0,0)',
        paper_bgcolor = 'rgba(0,0,0,0)',
        coloraxis_colorbar = dict(title = 'Renewable<br>Generation', ticksuffix = '%')
    )
    return fig

def make_fig5(countries, years):
    data = generated_differential(df, countries = countries, years = years)
    if data.empty or len(data.columns) < 3:
        return empty_figure(f'Change in Renewable Energy Share ({min(years)}-{max(years)})', 'No data for this selection')
    start_year, end_year = data.columns[1], data.columns[2]
    #dodge overlapping labels - plotly has no built-in label repulsion, so cluster
    #close endpoints and spread their labels to keep a minimum vertical gap
    gap = 4 #minimum label separation in %-points (~19px at this chart height)
    pts = data[['country', end_year]].sort_values(end_year, ascending = False).reset_index(drop = True)
    label_y = {}
    i = 0
    while i < len(pts):
        #grow the cluster while its span is too tight to fit all labels with gaps
        j = i
        while j + 1 < len(pts) and pts.loc[i, end_year] - pts.loc[j + 1, end_year] < gap * (j + 1 - i):
            j += 1
        cluster = pts.loc[i:j]
        #spread the cluster's labels evenly, centered on the cluster's midpoint
        mid = (pts.loc[i, end_year] + pts.loc[j, end_year]) / 2
        top = min(100, mid + gap * (len(cluster) - 1) / 2)
        top = max(top, gap * (len(cluster) - 1)) #keep the spread inside the 0-100 axis range
        for k, (_, r) in enumerate(cluster.iterrows()):
            label_y[r['country']] = top - k * gap
        i = j + 1
    fig = go.Figure()
    for _, row in data.iterrows():
        fig.add_trace(
            go.Scatter(
                x = [0, 1],
                y = [row[start_year], row[end_year]],
                mode = 'lines+markers',
                name = row['country'],
                cliponaxis = False,
            )
        )
    #country labels at the (possibly nudged) right endpoints - annotations render
    #in the margin layer, take a pixel offset, and hover ignores them
    for _, row in data.iterrows():
        fig.add_annotation(
            x = 1,
            y = label_y[row['country']],
            xref = 'x',
            yref = 'y',
            text = row['country'],
            showarrow = False,
            xanchor = 'left',
            xshift = 8, #nudge labels slightly right, away from the axis line
            font = dict(color = 'black')
        )
    #vertical line on the right side - the right axis of the slope chart
    fig.add_shape(
        type = 'line',
        xref = 'x', yref = 'paper',
        x0 = 1, y0 = 0,
        x1 = 1, y1 = 1,
        line = dict(color = 'black', width = 1)
    )

    #title must precede title_x - a title kwarg replaces the title object and would drop its x
    fig.update_layout(
        title = f'Change in Renewable Energy Share ({start_year}-{end_year})',
        title_x = 0.5,
        height = 650,
        plot_bgcolor = 'rgba(0,0,0,0)',
        paper_bgcolor = 'rgba(0,0,0,0)',
        showlegend = False,
        margin = dict(l = 80, r = 160, t = 100, b = 70),
        xaxis = dict(
            tickmode = 'array',
            tickvals = [0, 1],
            ticktext = [start_year, end_year],
            showgrid = False,
            showline = False,
            ticks = 'outside',
            ticklen = 15,
            tickcolor = 'rgba(0,0,0,0)',
            range = [0, 1]
        ),
        yaxis = dict(
            title = 'Renewable Energy Share (%)',
            range = [0, 100],
            ticksuffix = '%',
            dtick = 20,
            showline = True,
            linecolor = 'black',
            tickcolor = 'black'
        )
    )
    return fig


#prompt: instead of world, lets default the dashboard to the top 10 countries by electricity_generation_per_capita_kwh
#this was generated when I was still inlcuding World in the dataset; functionality with world removed shouldn't affect the code
#result:

# Shared controls at the top of the dashboard
# Default selection: the top 10 countries by total electricity generation (TWh).
# The 'World' aggregate is excluded from the dataset entirely - it has no ISO
# code, so it does not appear in the dropdown either.
# The latest years can be incomplete, so rank on the most recent year with at
# least 90% country coverage.
countries_only = df[df['country'] != 'World']
coverage = (countries_only.dropna(subset = ['electricity_generation_twh'])
              .groupby('year')['country'].nunique())
rank_year = coverage[coverage >= 0.9 * countries_only['country'].nunique()].index.max()
default_countries = (
    countries_only[countries_only['year'] == rank_year]
      .dropna(subset = ['electricity_generation_twh'])
      .nlargest(10, 'electricity_generation_twh')['country']
      .tolist()
)
country_options = sorted(df['country'].unique())
controls = html.Div(
    [
        html.Div(
            [
                html.Label('Countries', htmlFor = 'country-dropdown'),
                dcc.Dropdown(
                    id = 'country-dropdown',
                    options = country_options,
                    value = default_countries,
                    multi = True,
                    placeholder = 'Select countries…'
                ),
            ],
            style = {'flex': 1, 'paddingRight': '20px'}
        ),
        html.Div(
            [
                html.Label('Year range (the map uses the end year)', htmlFor = 'year-range'),
                dcc.RangeSlider(
                    id = 'year-range',
                    min = 1985,
                    max = int(df['year'].max()),
                    step = 1,
                    value = [1985, int(df['year'].max())],
                    allowCross = False,
                    marks = {y: str(y) for y in range(1985, int(df['year'].max()) + 1, 5)}
                ),
            ],
            style = {'flex': 1}
        ),
    ],
    style = {'display': 'flex', 'alignItems': 'flex-end', 'padding': '10px'}
)

app = Dash(__name__)
app.layout = html.Div(
    [
        controls,
        # tracks which fig1 legend entries are visible (double-click isolates a country)
        dcc.Store(id = 'fig1-legend-state', data = {}),
        # Row 1 - fig1 and fig2 side by side
        html.Div(
            [
                dcc.Graph(id = 'fig1-graph', style = {'flex': 1}),
                dcc.Graph(id = 'fig2-graph', style = {'flex': 1}),
            ],
            style = {'display': 'flex', 'gap': '10px'}
        ),
        # Row 2 - fig3 (map) and fig5 (slope chart) side by side
        html.Div(
            [
                dcc.Graph(id = 'fig3-graph', style = {'flex': 1}),
                dcc.Graph(id = 'fig5-graph', style = {'flex': 1}),
            ],
            style = {'display': 'flex', 'gap': '10px'}
        ),
    ]
)

# One callback links the shared controls to all four figures:
# countries and the year range window fig1/fig2/fig5; fig3 uses the range's end year.
@app.callback(
    Output('fig1-graph', 'figure'),
    Output('fig2-graph', 'figure'),
    Output('fig3-graph', 'figure'),
    Output('fig5-graph', 'figure'),
    Output('fig1-legend-state', 'data'),
    Input('country-dropdown', 'value'),
    Input('year-range', 'value'),
)
def update_figures(countries, year_range):
    if not countries:
        countries = default_countries
    year_range = sorted(year_range)
    years = range(year_range[0], year_range[1] + 1)  # full range for the time series
    year = year_range[1]                             # end year for the map
    return (
        make_fig1(countries, years),
        make_fig2(countries, years),
        make_fig3(year),
        make_fig5(countries, years),  # slope chart spans the range's start and end years
        None,  # a fresh fig1 shows every legend entry - reset visibility tracking
    )

# Double-clicking a fig1 legend entry isolates that country - Plotly hides the
# other traces and emits a restyle event, which Dash exposes as restyleData.
# Track visibility across events; fig2 then aggregates whatever fig1 still shows,
# so isolating a country turns fig2 into that country's energy mix. Double-click
# the entry again to bring every trace back and fig2 returns to the full selection.
@app.callback(
    Output('fig2-graph', 'figure', allow_duplicate = True),
    Output('fig1-legend-state', 'data', allow_duplicate = True),
    Input('fig1-graph', 'restyleData'),
    State('fig1-legend-state', 'data'),
    State('fig1-graph', 'figure'),
    State('country-dropdown', 'value'),
    State('year-range', 'value'),
    prevent_initial_call = True,
)
def focus_fig2(restyle, legend_state, fig1, countries, year_range):
    if restyle is None:
        raise PreventUpdate
    trace_names = [t['name'] for t in fig1['data']]
    state = legend_state if legend_state else {name: True for name in trace_names}
    # apply this legend event: restyle = [{'visible': value(s)}, [trace indices]]
    edits, indices = (restyle[0], restyle[1]) if len(restyle) >= 2 else ({}, [])
    vis = edits.get('visible') if isinstance(edits, dict) else None
    if vis is not None:
        for k, idx in enumerate(indices or []):
            if 0 <= idx < len(trace_names):
                value = vis[k] if isinstance(vis, list) and k < len(vis) else vis
                state[trace_names[idx]] = (value is True)
    visible = [name for name in trace_names if state.get(name, True)]
    year_range = sorted(year_range)
    years = range(year_range[0], year_range[1] + 1)
    if len(visible) < len(trace_names):
        # fig1 is showing a subset - focus fig2 on the same countries
        fig2 = make_fig2(visible, years)
        if len(visible) == 1:
            #retitle in place - update_layout(title=...) replaces the title object and drops its centering
            fig2.layout.title.text = f"{visible[0]}: Energy Generated by Source ({year_range[0]}-{year_range[1]})"
    else:
        fig2 = make_fig2(countries if countries else default_countries, years)
    return fig2, state

# Serve the interactive version on port 8051 (the static Step 2 app is on 8050)
# make_server gives a shutdown handle, so re-running this cell replaces the server cleanly
from werkzeug.serving import make_server
if 'dash_server' in globals() and dash_server is not None:
    dash_server.shutdown()
dash_server = make_server('0.0.0.0', 8051, app, threaded = True)
Thread(target = dash_server.serve_forever, daemon = True).start()
print('Interactive dashboard serving at http://127.0.0.1:8051')

<img src="./image_1794842890782.png" width="50%">

### Exploration narrative

The initial question I started with was How has energy generation changed over time, especially in relation to the advent of AI?  Initial exploration was through using total generation, but this was problematic because different countries need different amounts, so the figure 1 was changed to be using per-capita data, allowing for a better comparison.  This chart highlighted that many countries were missing data prior to 1985, prompting filtering to be adjusted.  Then by looking at the top 10 by total generation there are some interesting trends to see.
- China having steady growth while places Canada and Germany show decline.  
- South Korea's growth was a huge suprise as was the steady decline in the United States.  
- There does seem to be an inflection point for the US around 2020 suggesting generation per capita is trending upward.
- One country that wasn't in the top 10 by generation that was an interesting find was Iceland, being the largest generator per capita of the dataset

From there the next question I wanted to explore was: How has the breakdown of generation by source changed over time?.  Figure 2 was deveoloped to show the aggregated values of all the countries selected colored by source (oil, gas, solar, etc.).  Later functionality was added for interaction from figure 1; focusing a curve by double-clicking on it in the legend focuses figure 2 to that country to aid in exploring a single country easily.
- Coal is still a major contributor but growth is slowing
- Renewables are growing (band getting thicker)
- United States was primarily coal but has shifted to primarily gas
- Iceland is fully renewable

With the growth  trend in renewables, the next question I had was: How does each country compare in terms of their share of generation from renewables?  A choropleth chart was developed for figure 3 with a 0-100% color scale to show how each country compares in overall generation from renewable energy.  Going from red (low) to yellow (mid) to green (high) showed some unexpected places had high renewable energy generation.
- South America and South/Central Africa were regions that were unexpectedly high
- United states is behind China in renewables

Next is where I ran into some issues with trying to explore how different countries compare on renewables, fossil fuels and carbon emissions.  I initially developed figure 4, a parallel coordinates plot for this purpose with each line representing one of the selected countries.  You could see how a country that was high in renewables, would be low in fossil fuels and subsequently lower in emissions but getting the figure to distinguish which country was which on the plot was proving difficult so I pivoted to a different chart.

Ultimately the final chart, figure 5, was developed as a slope chart to provide a visual that provides a quick at-a-glance visual representation of how each countries renewable generation changed over the selected period.  This chart showed me some issues that coudl arise with some missing data.  I made a big assumption in the processing for the start point of the chart, that if the value is missing, it should be 0.  This processing highlighted that there was a significant amount of countries missing data for 2025, prompting another shift in filters.  I have not fully explored all options but am amware that there could be some more countries showing odd behavior based on this processing.  Here are some observations I made based ont this chart:
- Brazil, Canada, and Germany are leading the way in renewable growth, being nearly 0% in 1985
- Growth rate between United States, China, Japan and India are similar.
- If you filter down to the last 5 years, growth rate for most countries is fairly low with Canada showing decline.

## AI log and troubleshooting (10 pts)

Entry 1: Generate Basic Dash App
- Tool: Databricks Assistant (Genie Code)
- What I asked for / what I was trying to do: Using the code in this notebook, generate a dash app using the following layout: Row 1: Fig1, Fig2 Row 2: Fig3, Fig5
- What came back / what happened: a basic dash app was created
- What was wrong or could be better: the spacing of the figures was not defined and I hadn't defined any widths so it didn't look quite right
- How I noticed: ran the app locally and saw the figures weren't aligned neatly
- What I did instead: added another prompt to fix the spacing, which was successful

Entry 2: Add Interactivity
- Tool: Databricks Assistant (Genie Code)
- What I asked for / what I was trying to do: Add callbacks to link the figures
- What came back / what happened: Successfully generated the code for the controls, converted my figure code into functions and created callbacks
- What was wrong or could be better: the control for selecting year was just single value slider when I was expecting a range
- How I noticed: ran the new app locally and tested the interactivity
- What I did instead: added a new prompt 'adjust the year slider to a year-range control', which was successful

Entry 3: Assistant inserted a new cell under the wrong heading
- Tool: Databricks Assistant (Genie Code)
- What I asked for / what I was trying to do: add the linked-interactivity code
- What came back / what happened: the cell was inserted after the "Step 2 - Layout" markdown instead of the expected Step - 3  
- What was wrong or could be better: I didn't specify that it should be under Step 3
- How I noticed: checked the notebook's cell order after generation
- What I did instead: cut and moved the cell to the proper location

Entry 4: Change background color
- Tool: Databricks Assistant (Genie Code)
- What I asked for / what I was trying to do: update the background of the charts to transparent (or white if possible)
- What came back / what happened: successfully updated the backgrounds
- What was wrong or could be better: the axes disappeared on some of the figures
- How I noticed: re-deployed the updated app with the changes
- What I did instead: added a new prompt: with the change in background to white, the axes disappeared, need to see those in black

## Deployment instructions (10 pts)

The dashboard is deployed as a **Databricks App**


**Live URL:** https://siads-521-energy-dashboard-7474654444631410.aws.databricksapps.com

**Alternate:** https://siads-521-energy-dashboard-c3d248ea.plotly.app/

*Note: the app is OAuth-protected, so you must be logged in to a Databricks account (any workspace member with access) to view it - anonymous visitors are redirected to the login page.*

### How it runs

The app's source lives in the databricks workspace folder, `/Users/account/siads-521-energy-dashboard/`, containing four files:

* `app.py` - the full dashboard: data loading/cleaning (same steps as this notebook), the four figure builders, the layout, and the shared country/year-range callback. It exposes `server = app.server` as the web entrypoint.
* `app.yaml` - the run command: `gunicorn app:server --bind=0.0.0.0:8000` (gunicorn serves the Flask app that Dash wraps, on the port Databricks Apps expects)
* `requirements.txt` - `dash`, `pandas`, `plotly`, `gunicorn` (Databricks builds a fresh environment from this on every deploy)
* `owid_energy.csv` - the dataset, bundled with the app so it is self-contained

To limit data usage, two jobs were setup to run automatic scripts at certain times to start and stop the app at certain times.

*Note: Currently it is setup to be active between 05:00 and 20:00 CST/CDT due to limits on compute.*

### Steps to repeat it

Since the source files are copied over already created, repeating the deployment only requires creating the Databricks App from the existing folder and deploying it - no files need to be regenerated.

1. Create a workspace folder with the four files above (`app.py`, `app.yaml`, `requirements.txt`, and the CSV next to it - the app reads the CSV by relative path).
2. Open the Databricks Assistant (Genie Code) through the icon in the top right and prompt it to create and deploy the app from that folder as-is: 
   - `Create a Databricks App from the folder /Users/your_account/siads-521-energy-dashboard/ using the existing app.py, app.yaml, and requirements.txt, and deploy it.` Databricks builds a fresh environment from `requirements.txt`, creates the app serverless inside the workspace, and returns the live URL.
3. Open the returned URL while logged in to the workspace to verify the dashboard renders and the linked interactivity works.
4. (Optional) To limit data usage, set up scheduled jobs to start and stop the app at fixed times, as described above.